In [29]:
import os
import pandas as pd
import tensorflow as tf
from pathlib import Path

In [30]:
print(Path.cwd().parent)

/Users/finnerty/Documents/Swinburne/FinalSem/MachineSystems/AMLFacialRecognitionProject


In [32]:
PROJECT_ROOT =  Path.cwd().parent

DATASET = (
    PROJECT_ROOT
    / "data"
    / "11-785-fall-20-homework-2-part-2"
    / "antispoof"
)

TRAIN_IMG_DIR = DATASET / "train"
TEST_IMG_DIR = DATASET / "test"
VAL_IMG_DIR = DATASET / "valid"

TRAIN_CSV = os.path.join(TRAIN_IMG_DIR, "_classes.csv")
TEST_CSV = os.path.join(TEST_IMG_DIR, "_classes.csv")
VAL_CSV = os.path.join(VAL_IMG_DIR, "_classes.csv")

In [33]:
def load_labels(csv_path, image_dir):
    df = pd.read_csv(csv_path)

    image_paths = []
    labels = []

    for _, row in df.iterrows():
        filename = row["filename"]
        fake = row[" fake"]
        real = row[" real"]

        if fake == real:
            continue

        image_path = os.path.join(image_dir, filename)

        if not os.path.exists(image_path):
            continue

        label = 1 if real == 1 else 0

        image_paths.append(image_path)
        labels.append(label)

    return image_paths, labels

In [34]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def preprocess_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    return image, label


def create_dataset(image_paths, labels, shuffle=True):
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    dataset = dataset.map(preprocess_image)

    if shuffle:
        dataset = dataset.shuffle(1000)

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

In [ ]:
train_paths, train_labels = load_labels(TRAIN_CSV, TRAIN_IMG_DIR)
valid_paths, valid_labels = load_labels(VAL_CSV, VAL_IMG_DIR)
test_paths, test_labels = load_labels(TEST_CSV, TEST_IMG_DIR)

train_ds = create_dataset(train_paths, train_labels)
valid_ds = create_dataset(valid_paths, valid_labels, shuffle=False)
test_ds = create_dataset(test_paths, test_labels, shuffle=False)

print("train: ", len(train_paths))
print("val: ", len(valid_paths))
print("test: ", len(test_paths))

Train: 756
Valid: 129
Test: 96


In [ ]:
import numpy as np

print("train true: ", np.sum(train_labels))
print("train fake : ", len(train_labels) - np.sum(train_labels))

print("val true: ", np.sum(valid_labels))
print("val fake: ", len(valid_labels) - np.sum(valid_labels))

Train real: 341
Train fake: 415
Valid real: 67
Valid fake: 62


In [37]:
for i in range(10):
    print(train_paths[i], train_labels[i])

/Users/finnerty/Documents/Swinburne/FinalSem/MachineSystems/AMLFacialRecognitionProject/data/11-785-fall-20-homework-2-part-2/antispoof/train/527054_png.rf.75efd5941b9c3a495592efac26437119.jpg 1
/Users/finnerty/Documents/Swinburne/FinalSem/MachineSystems/AMLFacialRecognitionProject/data/11-785-fall-20-homework-2-part-2/antispoof/train/tumblr_n9slh4Z1uy1qg7epco1_640_jpg.rf.77d0dab348b7e8e2f6ce5f8c5f0b4d97.jpg 1
/Users/finnerty/Documents/Swinburne/FinalSem/MachineSystems/AMLFacialRecognitionProject/data/11-785-fall-20-homework-2-part-2/antispoof/train/frame-Data2-517_jpg.rf.7be95b4b61dd11de290e36ffccff6bc4.jpg 1
/Users/finnerty/Documents/Swinburne/FinalSem/MachineSystems/AMLFacialRecognitionProject/data/11-785-fall-20-homework-2-part-2/antispoof/train/392031_jpg.rf.79597b0904a8d4852976e75063bd71f5.jpg 0
/Users/finnerty/Documents/Swinburne/FinalSem/MachineSystems/AMLFacialRecognitionProject/data/11-785-fall-20-homework-2-part-2/antispoof/train/472559_jpg.rf.785c89a540c63144597a2bc3396b76f

In [58]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = True



model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),
    tf.keras.layers.Rescaling(1./255),

    data_augmentation,

    base_model,

    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [59]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [60]:
model.compile( 
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
    loss="binary_crossentropy", 
    metrics=["accuracy"] 
)

model.summary() 

history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=20,
    callbacks=[early_stop]
)

Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_7 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_11 (Sequential)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,340,033 (8.93 MB)

 Trainable params: 2,305,921 (8.80 MB)

 Non-trainable params: 34,112 (133.25 KB)

Epoch 1/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 28s 843ms/step - accuracy: 0.8386 - loss: 0.3560 - val_accuracy: 0.7364 - val_loss: 2.5120
Epoch 2/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 20s 805ms/step - accuracy: 0.9537 - loss: 0.1180 - val_accuracy: 0.7907 - val_loss: 1.3621
Epoch 3/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 20s 806ms/step - accuracy: 0.9590 - loss: 0.1206 - val_accuracy: 0.6047 - val_loss: 5.6358
Epoch 4/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 18s 721ms/step - accuracy: 0.9511 - loss: 0.1275 - val_accuracy: 0.7597 - val_loss: 2.5005


In [61]:
loss, accuracy = model.evaluate(test_ds)
print("accuracy: ", accuracy)
print("loss: ", loss)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 0.9792 - loss: 0.1233
accuracy:  0.9791666865348816
loss:  0.1233314573764801


In [51]:
model.save("./antispoof.keras")